# Self-Consistency: Majority Voting Over Reasoning Paths

Self-consistency (Wang et al. 2022) samples multiple reasoning paths from the model and takes the majority vote over final answers. It improves accuracy beyond greedy decoding with no model changes — purely a decoding strategy.

## The Core Insight

Greedy decoding commits to the single highest-probability reasoning path. But for complex problems, the highest-probability path is not always correct. Different valid reasoning approaches may all reach the correct answer via different chains.

Self-consistency exploits this: sample K diverse chains using temperature > 0, extract the final answer from each, and return the most common answer. Incorrect chains produce varied wrong answers; correct reasoning concentrates on the right answer.

The marginalizing effect: even if each individual chain is only 60% likely to be correct, if correct answers cluster and wrong answers scatter, majority voting over 10+ samples can reach 90%+ accuracy.

In [ ]:
from dataclasses import dataclass
from collections import Counter
from typing import Callable
import random

@dataclass
class ConsistencyResult:
    problem: str
    chains: list
    answers: list
    majority_answer: str
    confidence: float
    correct: bool
    n_samples: int

def sample_cot(model_fn: Callable, prompt: str, n: int, temperature: float, seed: int = 42) -> list:
    rng = random.Random(seed)
    chains = []
    for i in range(n):
        noise_seed = seed + i
        chain = model_fn(prompt, temperature=temperature, seed=noise_seed)
        chains.append(chain)
    return chains

def self_consistency(problem: str, model_fn: Callable, examples: list,
                     n_samples: int = 10, temperature: float = 0.7,
                     correct_answer: str = '') -> ConsistencyResult:
    import re
    prompt = build_cot_prompt(problem, examples)
    chains = sample_cot(model_fn, prompt, n_samples, temperature)
    answers = [extract_answer(c) for c in chains]
    counts = Counter(a for a in answers if a)
    majority = counts.most_common(1)[0][0] if counts else ''
    confidence = counts[majority] / n_samples if majority else 0.0
    correct = majority == str(correct_answer)
    return ConsistencyResult(
        problem=problem, chains=chains, answers=answers,
        majority_answer=majority, confidence=confidence,
        correct=correct, n_samples=n_samples
    )

def extract_answer(chain: str) -> str:
    import re
    match = re.search(r'(?:answer(?:\s+is)?|therefore)[:\s]+([\d,\.]+)', chain, re.I)
    if match:
        return match.group(1).replace(',', '')
    numbers = re.findall(r'[\d,]+(?:\.\d+)?', chain)
    return numbers[-1].replace(',','') if numbers else ''

def build_cot_prompt(problem, examples):
    parts = []
    for ex in examples:
        parts.append(f'Q: {ex["problem"]}')
        parts.append(f'A: {ex["chain"]}')
        parts.append(f'Answer: {ex["answer"]}\n')
    parts.append(f'Q: {problem}')
    parts.append('A:')
    return '\n'.join(parts)

# Mock model with temperature-controlled noise
def noisy_model(prompt, temperature=0.7, seed=42):
    rng = random.Random(seed)
    # 70% chance of correct reasoning at temp=0.7
    if rng.random() < (1 - temperature * 0.4):
        if 'eggs' in prompt:
            return 'Selling 45 from 120 leaves 75. Boxes: 75/15 = 5. The answer is 5.'
        return 'Working through this: the answer is 5.'
    else:
        # Wrong path
        wrong = rng.choice(['3', '7', '10', '4'])
        return f'I calculate the answer is {wrong}.'

examples = [
    {'problem': 'Store has 20 items, sells 8, gets 5 more. How many?', 'chain': '20-8=12, 12+5=17', 'answer': '17'},
]
problem = 'A farmer has 120 eggs. He sells 45 and packs the rest in boxes of 15. How many boxes?'
result = self_consistency(problem, noisy_model, examples, n_samples=15, correct_answer='5')

print(f'Problem: {result.problem}')
print(f'Sampled {result.n_samples} reasoning paths')
print(f'Answer distribution: {Counter(result.answers).most_common(5)}')
print(f'Majority answer: {result.majority_answer} (confidence: {result.confidence:.0%})')
print(f'Correct: {result.correct}')

## Agreement Scoring and Calibration

The majority vote confidence (fraction of samples agreeing) is a useful but noisy calibration signal:
- **High confidence (>80%)**: model is likely correct; safe to trust without verification
- **Medium confidence (50-80%)**: ambiguous; consider re-prompting with different framing or escalating to a larger model
- **Low confidence (<50%)**: model is confused; likely needs more context or a different approach

Confidence calibration varies by domain. Mathematical reasoning shows better calibration (high confidence → high accuracy) than commonsense reasoning.

In [ ]:
def calibration_analysis(results: list, n_bins: int = 5) -> list:
    bins = [[] for _ in range(n_bins)]
    for r in results:
        bin_idx = min(int(r.confidence * n_bins), n_bins - 1)
        bins[bin_idx].append(r.correct)
    analysis = []
    for i, b in enumerate(bins):
        if not b:
            continue
        low = i / n_bins
        high = (i + 1) / n_bins
        analysis.append({
            'confidence_range': f'{low:.1f}-{high:.1f}',
            'n': len(b),
            'accuracy': sum(b) / len(b),
            'mean_confidence': (low + high) / 2,
        })
    return analysis

# Simulate a calibration experiment
rng = random.Random(42)
simulated_results = []
for _ in range(100):
    conf = rng.uniform(0.3, 1.0)
    # Well-calibrated: accuracy tracks confidence
    correct = rng.random() < conf * 0.9
    simulated_results.append(type('R', (), {'confidence': conf, 'correct': correct})())

analysis = calibration_analysis(simulated_results)
print('Calibration analysis (self-consistency confidence vs accuracy):')
print(f'{'Confidence':>12} {'N':>5} {'Accuracy':>10} {'Calibrated?'}')
for row in analysis:
    gap = abs(row['accuracy'] - row['mean_confidence'])
    cal = 'Yes' if gap < 0.1 else 'No'
    print(f"{row['confidence_range']:>12} {row['n']:>5} {row['accuracy']:>9.1%}  {cal}")

## Cost vs Accuracy Tradeoff

Self-consistency multiplies inference cost by K (the number of samples). The accuracy-cost tradeoff follows diminishing returns:
- K=1: greedy decoding baseline
- K=5: significant improvement for most reasoning tasks
- K=10: strong performance, commonly used in papers
- K=20+: marginal improvement over K=10; rarely justified unless accuracy is critical

For production systems, adaptive sampling is more efficient: start with K=1, only sample more if confidence is below a threshold. This achieves near-K=10 accuracy at K=2-3 average cost.